# Activity #1 — Train Your Own Neural Tree

**Workshop:** From Black Boxes to Glass Boxes  
**Instructor:** Cagri Temel — Hezarfen LLC  
**Venue:** Washington State University · May 15, 2026

---

## What you will do (~55 min)

1. Load the NASA CMAPSS turbofan engine dataset
2. Explore sensor degradation patterns
3. Bin RUL into 3 classes (Critical · Caution · Healthy)
4. Train a **SoftDecisionTree** classifier
5. Evaluate accuracy + confusion matrix
6. **Traverse a prediction** — read the rule the model used (the heart of the workshop)

## Setup notes

- **Runtime:** keep on CPU. GPU is unnecessary and just adds queue time.
- If you see `Restart runtime` after install — click Restart, then run the next cell.
- Cells with `# TODO:` are for you to complete. The instructor's solution notebook is in the same repo.


## Step 0 — Install and import

Run this cell first. Takes ~30 seconds.

In [ ]:
!pip install -q neural-trees pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from neural_trees import SoftDecisionTree

print('All set.')

## Step 1 — Download the data

NASA CMAPSS FD001 — 100 turbofan engines simulated to failure under one operating condition.

In [ ]:
BASE = 'https://raw.githubusercontent.com/cgrtml/wsu-workshop-may15/main/data'
!wget -q -nc {BASE}/train_FD001.txt
!wget -q -nc {BASE}/test_FD001.txt
!wget -q -nc {BASE}/RUL_FD001.txt

import os
for f in ['train_FD001.txt', 'test_FD001.txt', 'RUL_FD001.txt']:
    print(f, '->', os.path.getsize(f), 'bytes')

## Step 2 — Load and structure

Each row = one timestep for one engine. Columns:
- `engine_id` (1–100)
- `cycle` (1, 2, 3, …)
- 3 operational settings
- 21 sensor channels


In [ ]:
cols = ['engine_id', 'cycle'] + [f'op{i}' for i in range(1, 4)] + [f's{i}' for i in range(1, 22)]

train = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)
print(train.shape, '->', train['engine_id'].nunique(), 'engines')
train.head()

## Step 3 — Compute RUL for each timestep

For training data: RUL = max_cycle_for_that_engine − current_cycle.

In [ ]:
max_cycles = train.groupby('engine_id')['cycle'].max().rename('max_cycle')
train = train.merge(max_cycles, on='engine_id')
train['RUL'] = train['max_cycle'] - train['cycle']
train['RUL'] = train['RUL'].clip(upper=125)  # standard practice
train[['engine_id', 'cycle', 'max_cycle', 'RUL']].head()

## Step 4 — Visualize one engine's degradation

Look at how a few sensors evolve as the engine approaches failure.

In [ ]:
engine = train[train['engine_id'] == 1]
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, sensor in zip(axes.flat, ['s2', 's7', 's11', 's14']):
    ax.plot(engine['cycle'], engine[sensor])
    ax.set_title(f'Engine 1 — {sensor}')
    ax.set_xlabel('cycle')
plt.tight_layout(); plt.show()

### TODO 1

Plot the same 4 sensors for engine 24. Do they degrade in the same direction?


In [ ]:
# TODO 1: plot s2, s7, s11, s14 for engine_id == 24
# Hint: copy the cell above and change the engine_id



## Step 5 — Bin RUL into 3 classes

We turn this into a 3-way classification problem because the explanations are far easier to read.

| Class | RUL range | Meaning |
|---|---|---|
| 0 — Critical | RUL < 30 | Schedule maintenance now |
| 1 — Caution | 30 ≤ RUL < 80 | Monitor closely |
| 2 — Healthy | RUL ≥ 80 | Operate normally |


In [ ]:
def bin_rul(rul):
    if rul < 30:
        return 0  # Critical
    elif rul < 80:
        return 1  # Caution
    return 2      # Healthy

train['health'] = train['RUL'].apply(bin_rul)
train['health'].value_counts().sort_index()

## Step 6 — Build the feature matrix

Drop sensors with no variance, drop operational settings (constant in FD001), keep 14 informative sensors.

In [ ]:
all_sensors = [f's{i}' for i in range(1, 22)]
informative = [s for s in all_sensors if train[s].std() > 0.001]
print('Informative sensors:', informative)

X = train[informative].values
y = train['health'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)
print('X shape:', X.shape, '· y shape:', y.shape)

### TODO 2

Make a stratified train / test split: 80% train, 20% test. Use `random_state=42` so we all get the same numbers.


In [ ]:
# TODO 2: create X_train, X_test, y_train, y_test
# Hint: from sklearn.model_selection import train_test_split — already imported.
# Use stratify=y so class proportions are preserved.

# X_train, X_test, y_train, y_test = ...
# print(X_train.shape, X_test.shape)

## Step 7 — Train the Soft Decision Tree

**This is the core of the workshop.** `SoftDecisionTree` is a fully differentiable tree from the `neural-trees` package. Internal nodes are sigmoid splits; leaves are class distributions.

Parameters:
- `depth=4` → 16 leaves
- `max_epochs=30` → trains for 30 epochs
- `penalty_coef=1e-3` → encourages balanced splits


### TODO 3

Train a `SoftDecisionTree` with the parameters above. Print the test-set accuracy.


In [ ]:
# TODO 3: train and score
# tree = SoftDecisionTree(depth=4, max_epochs=30, learning_rate=0.01, penalty_coef=1e-3, verbose=True)
# tree.fit(X_train, y_train)
# acc = (tree.predict(X_test) == y_test).mean()
# print(f'Test accuracy: {acc:.3f}')


## Step 8 — Confusion matrix and per-class report

In [ ]:
# Run this AFTER you complete TODO 3
y_pred = tree.predict(X_test)
labels = ['Critical', 'Caution', 'Healthy']

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.title('Confusion matrix'); plt.show()

print(classification_report(y_test, y_pred, target_names=labels))

## Step 9 — Traverse a prediction (the heart of the workshop)

Pick one engine reading. Ask the model: *which sensor at which threshold drove your decision?*

`get_split_weights()` returns the linear weight vector at every internal node. The argmax of |weight| tells you the dominant feature for that node.

In [ ]:
splits = tree.get_split_weights()
print(f'Number of internal nodes: {len(splits)}')
print(f'Each split has {splits[0].shape[0]} weights (one per sensor)')

# Most influential sensor per node
for i, w in enumerate(splits):
    dominant = np.argmax(np.abs(w))
    print(f'  Node {i:2d} → dominant sensor: {informative[dominant]:4s} (weight={w[dominant]:+.3f})')

### TODO 4

Pick test sample at index `17`. Predict its class. Print:
1. The predicted class (Critical / Caution / Healthy)
2. The class probabilities
3. The dominant sensor at the root node — the first thing the model looks at


In [ ]:
# TODO 4: traverse one prediction
idx = 17
x = X_test[idx]
true = labels[y_test[idx]]

# Your code:
# pred_class = ...
# proba = ...
# root_dominant_sensor = ...

# print(f'True class:      {true}')
# print(f'Predicted class: {pred_class}')
# print(f'Probabilities:   {proba}')
# print(f'Root node looks at: {root_dominant_sensor}')


## Step 10 — Discussion

Talk to the person next to you for 3 minutes:

1. Which sensor dominated at the root? Does that match what you saw in Step 4?
2. If a regulator asked you *"why did you flag this engine as Critical?"*, what would you tell them?
3. What's one thing this model **cannot** tell you?

---

**Save your notebook** (File → Save in Drive). You will use the trained `tree` object in Activity #2.
